In [ ]:
# ============================================================
# 1. Install required packages
# ============================================================

!pip install -q langchain langchain-community chromadb sentence-transformers openai gradio

In [3]:
# ============================================================
# 2. Import libraries
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


import gradio as gr
from openai import OpenAI

In [4]:
# ============================================================
# 3. Upload NASA C-MAPSS dataset zip file
# Upload CMAPSSData.zip when Colab asks
# ============================================================

from google.colab import files
uploaded = files.upload()

!unzip -q CMAPSSData.zip -d extracted_data

Saving CMAPSSData.zip to CMAPSSData.zip


In [36]:
# ============================================================
# 4. Load FD001-FD004 datasets
# ============================================================

path = "extracted_data/"

columns = [
    'unit','cycle',
    'op1','op2','op3',
    's1','s2','s3','s4','s5',
    's6','s7','s8','s9','s10',
    's11','s12','s13','s14','s15',
    's16','s17','s18','s19','s20','s21'
]

train_files = ["train_FD001.txt", "train_FD002.txt", "train_FD003.txt", "train_FD004.txt"]
test_files  = ["test_FD001.txt",  "test_FD002.txt",  "test_FD003.txt",  "test_FD004.txt"]
rul_files   = ["RUL_FD001.txt",   "RUL_FD002.txt",   "RUL_FD003.txt",   "RUL_FD004.txt"]

train_list = []
test_list = []
rul_list = []

for i in range(4):
    dataset_id = i + 1

    train_temp = pd.read_csv(path + train_files[i], sep=r"\s+", header=None)
    test_temp  = pd.read_csv(path + test_files[i], sep=r"\s+", header=None)
    rul_temp   = pd.read_csv(path + rul_files[i], sep=r"\s+", header=None)

    train_temp.columns = columns
    test_temp.columns = columns

    train_temp["dataset"] = dataset_id
    test_temp["dataset"] = dataset_id
    rul_temp["dataset"] = dataset_id

    train_list.append(train_temp)
    test_list.append(test_temp)
    rul_list.append(rul_temp)

train_df = pd.concat(train_list, ignore_index=True)
test_df = pd.concat(test_list, ignore_index=True)
RUL_df = pd.concat(rul_list, ignore_index=True)

train_df["engine_id"] = train_df["dataset"].astype(str) + "_" + train_df["unit"].astype(str)
test_df["engine_id"] = test_df["dataset"].astype(str) + "_" + test_df["unit"].astype(str)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("RUL shape:", RUL_df.shape)

Train shape: (160359, 28)
Test shape: (104897, 28)
RUL shape: (707, 2)


In [37]:
# ============================================================
# 5. Create capped RUL labels
# ============================================================

RUL_CAP = 125

max_cycle = train_df.groupby("engine_id")["cycle"].max().reset_index()
max_cycle.columns = ["engine_id", "max_cycle"]

train_df = train_df.merge(max_cycle, on="engine_id", how="left")
train_df["RUL"] = train_df["max_cycle"] - train_df["cycle"]
train_df["RUL"] = train_df["RUL"].clip(upper=RUL_CAP)
train_df = train_df.drop(columns=["max_cycle"])

train_df.head()

,unit,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s15,s16,s17,s18,s19,s20,s21,dataset,engine_id,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,8.4195,0.03,392,2388,100.0,39.06,23.4190,1,1_1,125
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,8.4318,0.03,392,2388,100.0,39.00,23.4236,1,1_1,125
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,8.4178,0.03,390,2388,100.0,38.95,23.3442,1,1_1,125
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,8.3682,0.03,392,2388,100.0,38.88,23.3739,1,1_1,125
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,8.4294,0.03,393,2388,100.0,38.90,23.4044,1,1_1,125


In [38]:
# ============================================================
# 6. Feature selection and scaling
# ============================================================

feature_cols = [
    col for col in train_df.columns
    if col not in ["unit", "cycle", "RUL", "dataset", "engine_id"]
]

scaler = MinMaxScaler()

train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols] = scaler.transform(test_df[feature_cols])

print("Number of features:", len(feature_cols))
print(feature_cols)

Number of features: 24
['op1', 'op2', 'op3', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21']


In [39]:
# ============================================================
# 7. Create LSTM sequences
# ============================================================

SEQ_LEN = 30

def create_train_sequences(df, sequence_length, feature_cols):
    X = []
    y = []

    for engine in df["engine_id"].unique():
        engine_df = df[df["engine_id"] == engine].sort_values("cycle")

        features = engine_df[feature_cols].values
        rul = engine_df["RUL"].values

        for i in range(len(engine_df) - sequence_length + 1):
            X.append(features[i:i + sequence_length])
            y.append(rul[i + sequence_length - 1])

    return np.array(X), np.array(y)


def create_test_sequences(df, sequence_length, feature_cols):
    X = []

    for engine in df["engine_id"].unique():
        engine_df = df[df["engine_id"] == engine].sort_values("cycle")
        features = engine_df[feature_cols].values

        if len(engine_df) >= sequence_length:
            X.append(features[-sequence_length:])
        else:
            pad_length = sequence_length - len(engine_df)
            pad = np.repeat(features[0].reshape(1, -1), pad_length, axis=0)
            padded_features = np.vstack((pad, features))
            X.append(padded_features)

    return np.array(X)


X_train, y_train = create_train_sequences(train_df, SEQ_LEN, feature_cols)
X_test = create_test_sequences(test_df, SEQ_LEN, feature_cols)
y_test = np.minimum(RUL_df[0].values, RUL_CAP)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (139798, 30, 24)
y_train: (139798,)
X_test: (707, 30, 24)
y_test: (707,)


In [40]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout

model = Sequential([
    Input(shape=(SEQ_LEN, X_train.shape[2])),

    LSTM(64, return_sequences=True),
    Dropout(0.2),

    LSTM(32),
    Dropout(0.2),

    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 87s 46ms/step - loss: 2093.3315 - mae: 39.5379 - val_loss: 1786.3900 - val_mae: 38.4118
Epoch 2/10
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 83s 47ms/step - loss: 1801.2455 - mae: 37.5348 - val_loss: 1749.1356 - val_mae: 37.7163
Epoch 3/10
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 80s 46ms/step - loss: 1472.6605 - mae: 32.7199 - val_loss: 762.1749 - val_mae: 22.9052
Epoch 4/10
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 80s 46ms/step - loss: 711.1695 - mae: 21.0100 - val_loss: 639.3132 - val_mae: 20.5549
Epoch 5/10
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 80s 46ms/step - loss: 587.1935 - mae: 18.9008 - val_loss: 578.4910 - val_mae: 18.8997
Epoch 6/10
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 82s 47ms/step - loss: 527.2633 - mae: 17.8237 - val_loss: 475.3632 - val_mae: 17.3516
Epoch 7/10
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 81s 46ms/step - loss: 475.5645 - mae: 16.8648 - val_loss: 521.2950 - val_mae: 17.2516
Epoch 8/10
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 81s 46ms/step - loss: 425.0853 - mae: 15.8511 - val_loss:

In [41]:
# ============================================================
# 9. Evaluate RUL model
# ============================================================

predictions = model.predict(X_test).flatten()
predictions = np.maximum(predictions, 0)

rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)

print("Test RMSE:", rmse)
print("Test MAE:", mae)

test_engines = test_df.groupby("engine_id").tail(1).copy()

results = pd.DataFrame({
    "dataset": test_engines["dataset"].values,
    "unit": test_engines["unit"].values,
    "engine_id": test_engines["engine_id"].values,
    "True_RUL": y_test,
    "Predicted_RUL": predictions
})

results.head(20)

23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step
Test RMSE: 18.933796435499605
Test MAE: 14.740964889526367


,dataset,unit,engine_id,True_RUL,Predicted_RUL
0,1,1,1_1,112,116.505234
1,1,2,1_2,98,115.736725
2,1,3,1_3,69,65.117111
3,1,4,1_4,82,94.323692
4,1,5,1_5,91,112.929283
5,1,6,1_6,93,114.588074
6,1,7,1_7,91,108.912506
7,1,8,1_8,95,112.318100
8,1,9,1_9,111,114.797417
9,1,10,1_10,96,97.631912


In [42]:
# ============================================================
# 11. Create engineering documents for RAG
# ============================================================

os.makedirs("engineering_docs", exist_ok=True)

maintenance_manual = """
Aircraft Engine Maintenance Guide:

If predicted Remaining Useful Life is below 30 cycles, the engine should be flagged for urgent inspection.
If predicted Remaining Useful Life is between 30 and 80 cycles, preventive maintenance should be scheduled.
If predicted Remaining Useful Life is above 80 cycles, the engine can continue operating with regular monitoring.

Rapid reduction in RUL may indicate compressor degradation, turbine wear, bearing deterioration, or abnormal thermal loading.
High exhaust gas temperature, pressure ratio drop, vibration increase, and abnormal fuel flow are common indicators of engine health degradation.
"""

fault_guide = """
Aircraft Engine Fault Diagnosis Guide:

Compressor degradation is commonly associated with reduced pressure ratio, reduced airflow capacity, and increased exhaust gas temperature.
Turbine degradation may occur due to thermal fatigue, blade erosion, material creep, or cooling inefficiency.
Bearing faults may be associated with vibration increase, temperature rise, and mechanical efficiency loss.

Machine learning-based prognostics can estimate Remaining Useful Life using historical time-series sensor data.
LSTM models are suitable for engine health monitoring because they capture temporal degradation patterns.
"""

cfd_summary = """
CFD and Thermal-Fluid Engineering Summary:

High inlet temperature, reduced cooling flow, and high thermal gradients can increase turbine blade thermal loading.
Poor airflow distribution may cause hot spots and accelerate component degradation.
Thermal-fluid simulation results can support AI reasoning by identifying regions of high temperature, pressure loss, and reduced cooling effectiveness.

CFD outputs such as temperature contours, pressure drop, velocity distribution, and heat transfer rate can be integrated with sensor-based predictive maintenance models.
"""

digital_twin_doc = """
Digital Twin Synchronization Guide:

A digital twin maintains a continuously updated virtual representation of the physical engine system.
Sensor data, predicted RUL, operating conditions, maintenance history, and simulation outputs can be synchronized into the digital twin.
The digital twin supports condition-based maintenance by comparing current engine behavior against expected healthy behavior.

When AI predictions and digital twin state both indicate degradation, maintenance confidence increases.
"""

with open("engineering_docs/maintenance_manual.txt", "w") as f:
    f.write(maintenance_manual)

with open("engineering_docs/fault_guide.txt", "w") as f:
    f.write(fault_guide)

with open("engineering_docs/cfd_summary.txt", "w") as f:
    f.write(cfd_summary)

with open("engineering_docs/digital_twin_doc.txt", "w") as f:
    f.write(digital_twin_doc)

print("Engineering documents created.")

Engineering documents created.


In [43]:
# ============================================================
# 12. Build RAG vector database
# ============================================================

from langchain_core.documents import Document

docs = []

for filename in os.listdir("engineering_docs"):
    if filename.endswith(".txt"):
        with open(os.path.join("engineering_docs", filename), "r") as f:
            text = f.read()
            docs.append(Document(page_content=text, metadata={"source": filename}))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="aerospace_vector_db"
)

print("RAG vector database created.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RAG vector database created.


In [44]:
# ============================================================
# 13. Retrieve engineering evidence
# ============================================================

def retrieve_engineering_evidence(query, k=3):
    retrieved_docs = vector_db.similarity_search(query, k=k)

    evidence = ""
    sources = []

    for doc in retrieved_docs:
        evidence += doc.page_content + "\n\n"
        sources.append(doc.metadata["source"])

    return evidence, sources


query = "engine has low RUL and high temperature"
evidence, sources = retrieve_engineering_evidence(query)

print("Retrieved Sources:", sources)
print(evidence)

Retrieved Sources: ['maintenance_manual.txt', 'maintenance_manual.txt', 'fault_guide.txt']
Rapid reduction in RUL may indicate compressor degradation, turbine wear, bearing deterioration, or abnormal thermal loading.
High exhaust gas temperature, pressure ratio drop, vibration increase, and abnormal fuel flow are common indicators of engine health degradation.

Rapid reduction in RUL may indicate compressor degradation, turbine wear, bearing deterioration, or abnormal thermal loading.
High exhaust gas temperature, pressure ratio drop, vibration increase, and abnormal fuel flow are common indicators of engine health degradation.

Aircraft Engine Fault Diagnosis Guide:

Compressor degradation is commonly associated with reduced pressure ratio, reduced airflow capacity, and increased exhaust gas temperature.
Turbine degradation may occur due to thermal fatigue, blade erosion, material creep, or cooling inefficiency.
Bearing faults may be associated with vibration increase, temperature ris

In [45]:
# ============================================================
# 14. Create digital twin state
# ============================================================

digital_twin_state = pd.DataFrame({
    "engine_id": results["engine_id"],
    "predicted_rul": results["Predicted_RUL"],
    "true_rul": results["True_RUL"],
    "status": np.where(
        results["Predicted_RUL"] < 30,
        "Urgent Inspection",
        np.where(results["Predicted_RUL"] < 80, "Preventive Maintenance", "Healthy")
    )
})

digital_twin_state.head(20)

,engine_id,predicted_rul,true_rul,status
0,1_1,116.505234,112,Healthy
1,1_2,115.736725,98,Healthy
2,1_3,65.117111,69,Preventive Maintenance
3,1_4,94.323692,82,Healthy
4,1_5,112.929283,91,Healthy
5,1_6,114.588074,93,Healthy
6,1_7,108.912506,91,Healthy
7,1_8,112.318100,95,Healthy
8,1_9,114.797417,111,Healthy
9,1_10,97.631912,96,Healthy


In [58]:
!pip install -q google-genai

In [62]:
import os
from google import genai

os.environ["GEMINI_API_KEY"] = "AQ.Ab8RN6J1km5MH9TEKxwHdXP1vB3dFpPY3INJBr2MI_-fwsTqHw"

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [65]:
def generate_copilot_response(engine_id, user_question):
    engine_row = digital_twin_state[digital_twin_state["engine_id"] == engine_id]

    if engine_row.empty:
        return "Engine ID not found."

    predicted_rul = float(engine_row["predicted_rul"].values[0])
    status = engine_row["status"].values[0]

    evidence, sources = retrieve_engineering_evidence(user_question, k=3)

    if predicted_rul < 30:
        recommendation = "Urgent inspection is required before continued operation."
        condition = "Critical degradation risk"
    elif predicted_rul < 80:
        recommendation = "Schedule preventive maintenance and continue close monitoring."
        condition = "Moderate degradation"
    else:
        recommendation = "Engine appears healthy. Continue normal monitoring."
        condition = "Healthy operating condition"

    answer = f"""
Generative AI Aerospace Copilot Response

Engine ID: {engine_id}
Predicted RUL: {predicted_rul:.2f} cycles
Digital Twin Status: {status}

1. Engine Health Condition:
{condition}

2. Maintenance Recommendation:
{recommendation}

3. Engineering Evidence:
{evidence}

4. Explainable Reasoning:
The recommendation is based on the predicted Remaining Useful Life, digital twin status,
and retrieved engineering maintenance rules. Lower RUL values indicate higher probability
of component degradation such as compressor wear, turbine thermal fatigue, or bearing deterioration.

5. Human-in-the-Loop Note:
A maintenance engineer should verify this recommendation using inspection records,
sensor trends, and operational history before final action.

Retrieved Sources: {", ".join(sources)}
"""

    return answer

In [66]:
sample_engine_id = digital_twin_state["engine_id"].iloc[0]

response = generate_copilot_response(
    engine_id=sample_engine_id,
    user_question="What maintenance action should be taken for this engine?"
)

print(response)


Generative AI Aerospace Copilot Response

Engine ID: 1_1
Predicted RUL: 116.51 cycles
Digital Twin Status: Healthy

1. Engine Health Condition:
Healthy operating condition

2. Maintenance Recommendation:
Engine appears healthy. Continue normal monitoring.

3. Engineering Evidence:
Aircraft Engine Maintenance Guide:

If predicted Remaining Useful Life is below 30 cycles, the engine should be flagged for urgent inspection.
If predicted Remaining Useful Life is between 30 and 80 cycles, preventive maintenance should be scheduled.
If predicted Remaining Useful Life is above 80 cycles, the engine can continue operating with regular monitoring.

Aircraft Engine Maintenance Guide:

If predicted Remaining Useful Life is below 30 cycles, the engine should be flagged for urgent inspection.
If predicted Remaining Useful Life is between 30 and 80 cycles, preventive maintenance should be scheduled.
If predicted Remaining Useful Life is above 80 cycles, the engine can continue operating with regula